# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahasuhail27-cyber/AI-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

## 1. Unit of Analysis + Time Window

**One row means:**  
One row represents the daily performance of a single content item (`content_hash_id`) for a single client (`client_hash_id`) on one report date (`report_date`).

**Table(s) used:**  
- `fact_content_daily_performance` (primary table)

**Time window:**  
This notebook uses the March 2026 partition (`month = '2026-03'`), which is a mid-panel month recommended by the project.

**Prediction / Ranking Goal:**  
Rank content items based on historical performance so they can be prioritized for optimization.

**Deliberately excluded:**  
`client_hash_id` and `content_hash_id` are used only as identifiers. Future information and label-derived fields are excluded to prevent data leakage.


In [6]:
grain_check = con.sql("""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS duplicate_rows
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
LIMIT 10
""").df()

grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,duplicate_rows


In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
!pip -q install duckdb pyarrow huggingface_hub pandas

In [1]:
from google.colab import userdata

token = userdata.get("HF_TOKEN")

print(len(token))
print(token.startswith("hf_"))

37
True


In [2]:
from huggingface_hub import HfApi
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN").strip()

api = HfApi(token=HF_TOKEN)

print(api.whoami())

{'type': 'user', 'id': '6a5bbf0fd9eab5215f57c4b7', 'name': 'mahasuhail', 'fullname': 'Maha Suhail', 'email': 'mahasuhail27@gmail.com', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1785542400, 'isPro': False, 'avatarUrl': '/avatars/1e190cfcdbeb1105736c4bbbc96a6bdf.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'FlyRank-HF', 'role': 'read', 'createdAt': '2026-07-30T16:45:37.702Z'}}}


In [3]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN").strip()

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

print("✅ DuckDB connected")

✅ DuckDB connected


In [5]:
print(preview.columns.tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


## 2. Fields: feature / label / context / excluded

## 2. Data Contract

### Features
Historical performance metrics that are available before making an optimization decision.

### Label / Proxy
Optimization priority (ranking target) based on future performance.

### Context
- report_date
- client_hash_id
- content_hash_id

These fields identify records and support grouping or joining but are not used as model features.

### Excluded
Identifiers (`client_hash_id`, `content_hash_id`) and any future or label-derived information are excluded to avoid leakage.


In [4]:
preview = con.sql("""
SELECT *
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
LIMIT 5
""").df()

preview

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## 3. Verify it with queries (grain, counts, missing values, windows)

### Query 1 – Grain Check
This query verifies that one row represents one unique combination of `report_date`, `client_hash_id`, and `content_hash_id`.

**Result:** No duplicate rows were returned, confirming the expected grain.

### Query 2 – Row Count and Date Window
This query confirms the number of rows and the reporting period.

**Result:**
- Total rows: **9,841,378**
- First date: **2026-03-01**
- Last date: **2026-03-31**

### Query 3 – Availability Check
This query verifies data availability using the `IS TRUE` condition.

**Result:**
- Total rows: **9,841,378**
- GSC available: **3,611,061**
- GA4 available: **413,966**

These checks confirm that not every row contains Search Console or Google Analytics data, so availability flags should be considered during feature engineering.

In [8]:
availability = con.sql("""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
""").df()

availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available,ga4_available
0,9841378,3611061.0,413966.0


In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
summary = con.sql("""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
""").df()

summary

,total_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


## 4. Data limits

This dataset has several limitations that should be considered before modeling.

- Different clients have different amounts of historical data, so the dataset is not balanced across clients.
- Many rows do not contain Google Search Console (GSC) or Google Analytics (GA4) data because the data was not available at that time.
- This notebook only uses the March 2026 partition, so the analysis does not represent the entire warehouse.
- Client IDs and Content IDs are identifiers only and should not be used as machine learning features.
- Future information or label-derived fields must be excluded because they would cause data leakage and produce unrealistic model performance.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Demonstrate one data limitation:
# Not every row has GSC or GA4 data available.

limits = con.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS FALSE) AS missing_gsc,
    COUNT(*) FILTER (WHERE ga4_data_available IS FALSE) AS missing_ga4
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
""").df()

limits

,total_rows,missing_gsc,missing_ga4
0,9841378,6230317,6408671


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.